In [0]:
%pip install pandas numpy scipy
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
import mlflow
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings("ignore")

spark = SparkSession.builder.getOrCreate()

print("Imports done.")

Imports done.


In [0]:
GERMAN_PATH = "/Volumes/workspace/default/hackbricks/german_credit_data.csv"

# German Credit column names (from UCI documentation)
german_col_names = [
    "checking_account_status",    # Status of existing checking account
    "duration_months",            # Duration of credit in months
    "credit_history",             # Credit history
    "purpose",                    # Purpose of loan
    "credit_amount",              # Credit amount (loan amount)
    "savings_account",            # Savings account / bonds
    "employment_since",           # Present employment since
    "installment_rate",           # Installment rate (% of disposable income)
    "personal_status_sex",        # Personal status and sex
    "other_debtors",              # Other debtors / guarantors
    "present_residence_since",    # Present residence since (years)
    "property",                   # Property owned
    "age",                        # Age in years
    "other_installment_plans",    # Other installment plans
    "housing",                    # Housing type
    "existing_credits",           # Number of existing credits
    "job",                        # Job type
    "liable_people",              # Number of people liable to provide maintenance
    "telephone",                  # Telephone
    "foreign_worker",             # Foreign worker
    "target"                      # 1 = Good credit, 2 = Bad credit
]

german_df = pd.read_csv(GERMAN_PATH, sep=",", header=None, names=german_col_names)

print(f"German Credit data loaded. Shape: {german_df.shape}")
print(f"Columns: {list(german_df.columns)}")
display(german_df.head())

German Credit data loaded. Shape: (1001, 21)
Columns: ['checking_account_status', 'duration_months', 'credit_history', 'purpose', 'credit_amount', 'savings_account', 'employment_since', 'installment_rate', 'personal_status_sex', 'other_debtors', 'present_residence_since', 'property', 'age', 'other_installment_plans', 'housing', 'existing_credits', 'job', 'liable_people', 'telephone', 'foreign_worker', 'target']


checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate,personal_status_sex,other_debtors,present_residence_since,property,age,other_installment_plans,housing,existing_credits,job,liable_people,telephone,foreign_worker,target
Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,null,null,null,null,null,null,null,null,null,null,null,null
67,male,2,own,null,little,1169,6,radio/TV,null,null,null,null,null,null,null,null,null,null,null,null
22,female,2,own,little,moderate,5951,48,radio/TV,null,null,null,null,null,null,null,null,null,null,null,null
49,male,1,own,little,null,2096,12,education,null,null,null,null,null,null,null,null,null,null,null,null
45,male,2,free,little,little,7882,42,furniture/equipment,null,null,null,null,null,null,null,null,null,null,null,null


In [0]:
# Load the original training data
train_df = pd.read_csv("/Volumes/workspace/default/hackbricks/application_train.csv")

print(f"Home Credit training data loaded. Shape: {train_df.shape}")

Home Credit training data loaded. Shape: (307511, 122)


In [0]:
# Extract the 5 comparable features from Home Credit and rename them to the common names for comparison with features from German credit
home_credit_features = {
    'age':              train_df['DAYS_BIRTH'].abs() / 365,     # Convert days to years
    'credit_amount':    train_df['AMT_CREDIT'],
    'duration_months':  train_df['CNT_PAYMENT'] if 'CNT_PAYMENT' in train_df.columns
                        else train_df['AMT_CREDIT'] / train_df['AMT_ANNUITY'],
    'installment_rate': (train_df['AMT_ANNUITY'] / train_df['AMT_INCOME_TOTAL'] * 100).clip(0, 100),
    'existing_credits': train_df['AMT_REQ_CREDIT_BUREAU_YEAR'].fillna(0)
}

german_features = {
    'age':              pd.to_numeric(german_df['age'], errors='coerce'),
    'credit_amount':    pd.to_numeric(german_df['credit_amount'], errors='coerce'),
    'duration_months':  pd.to_numeric(german_df['duration_months'], errors='coerce'),
    'installment_rate': (pd.to_numeric(german_df['installment_rate'], errors='coerce') * 12).clip(0, 100),
    'existing_credits': pd.to_numeric(german_df['existing_credits'], errors='coerce')
}

print("Feature mapping complete.")
print("\nHome Credit feature stats:")
for name, series in home_credit_features.items():
    print(f"  {name}: mean={series.mean():.2f}, std={series.std():.2f}")

print("\nGerman Credit feature stats:")
for name, series in german_features.items():
    print(f"  {name}: mean={series.mean():.2f}, std={series.std():.2f}")

Feature mapping complete.

Home Credit feature stats:
  age: mean=43.94, std=11.96
  credit_amount: mean=599026.00, std=402490.78
  duration_months: mean=21.61, std=7.82
  installment_rate: mean=18.09, std=9.44
  existing_credits: mean=1.64, std=1.86

German Credit feature stats:
  age: mean=nan, std=nan
  credit_amount: mean=nan, std=nan
  duration_months: mean=nan, std=nan
  installment_rate: mean=97.44, std=8.46
  existing_credits: mean=nan, std=nan


In [0]:
def calculate_psi(expected_series, actual_series, n_bins=10):
    """
    Calculate Population Stability Index between two distributions.

    Parameters:
        expected_series: The training data distribution (Home Credit)
        actual_series:   The current/new data distribution (German Credit)
        n_bins:          Number of bins to split data into (default 10)

    Returns:
        psi_value:   Single PSI score
        psi_df:      DataFrame with bin-level details
    """

    # Remove nulls
    expected_series = expected_series.dropna()
    actual_series   = actual_series.dropna()

    # Define bin edges based on training data (expected)
    # We use quantile-based bins so each bin has roughly equal # of training samples
    breakpoints = np.unique(
        np.percentile(expected_series, np.linspace(0, 100, n_bins + 1))
    )

    # If all values are the same (no variation), PSI can't be computed
    if len(breakpoints) < 2:
        return 0.0, pd.DataFrame()

    # Count samples in each bin for both datasets
    expected_counts = pd.cut(expected_series, bins=breakpoints, include_lowest=True).value_counts().sort_index()
    actual_counts   = pd.cut(actual_series,   bins=breakpoints, include_lowest=True).value_counts().sort_index()

    # Convert counts to percentages
    expected_pct = expected_counts / len(expected_series)
    actual_pct   = actual_counts   / len(actual_series)

    # Avoid division by zero or log(0) — replace 0% with a tiny number
    expected_pct = expected_pct.replace(0, 0.0001)
    actual_pct   = actual_pct.replace(0, 0.0001)

    # Calculate PSI contribution for each bin
    psi_contributions = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

    # Total PSI
    psi_value = psi_contributions.sum()

    # Build a nice detail table
    psi_df = pd.DataFrame({
        'bin':                expected_pct.index.astype(str),
        'expected_pct':       (expected_pct * 100).round(2),
        'actual_pct':         (actual_pct * 100).round(2),
        'psi_contribution':   psi_contributions.round(4)
    })

    return round(psi_value, 4), psi_df


def interpret_psi(psi_value):
    """Return a human-readable interpretation of PSI value."""
    if psi_value < 0.1:
        return "STABLE - No action needed"
    elif psi_value < 0.2:
        return "WARNING - Monitor closely"
    else:
        return "DRIFT DETECTED - Retrain model"


print("PSI functions defined.")

PSI functions defined.


In [0]:
psi_results = []

print("Computing PSI for each feature...\n")
print(f"{'Feature':<22} {'PSI':>8}  Status")
print("-" * 50)

for feature_name in home_credit_features.keys():

    expected = home_credit_features[feature_name]
    actual   = german_features[feature_name]

    psi_value, psi_detail = calculate_psi(expected, actual, n_bins=10)
    status = interpret_psi(psi_value)

    # Store result
    psi_results.append({
        'feature_name':  feature_name,
        'psi_value':     psi_value,
        'status':        status,
        'threshold':     0.2,
        'drift_detected': psi_value > 0.2,
        'n_expected':    len(expected),
        'n_actual':      len(actual)
    })

    print(f"{feature_name:<22} {psi_value:>8.4f}  {status}")

print("\nPSI computation complete!")

# Check if any feature has drifted
drifted_features = [r for r in psi_results if r['drift_detected']]
print(f"\nFeatures with drift (PSI > 0.2): {len(drifted_features)} out of {len(psi_results)}")

if drifted_features:
    print("Drift detected in:")
    for r in drifted_features:
        print(f"  - {r['feature_name']}: PSI = {r['psi_value']}")


Computing PSI for each feature...

Feature                     PSI  Status
--------------------------------------------------
age                      0.0000  STABLE - No action needed
credit_amount            0.0000  STABLE - No action needed
duration_months          0.0000  STABLE - No action needed
installment_rate         8.2832  DRIFT DETECTED - Retrain model
existing_credits         0.0000  STABLE - No action needed

PSI computation complete!

Features with drift (PSI > 0.2): 1 out of 5
Drift detected in:
  - installment_rate: PSI = 8.2832


In [0]:
from datetime import datetime

# Add timestamp to know when this drift check was run
for r in psi_results:
    r['computed_at'] = datetime.now().isoformat()
    r['batch_name']  = 'german_credit_simulation'

psi_results_df = pd.DataFrame(psi_results)

print("PSI Results DataFrame:")
display(psi_results_df)

# Create the database if it doesn't exist
spark.sql("CREATE DATABASE IF NOT EXISTS hackbricks")

# Convert to Spark DataFrame and write as Delta table
spark_psi_df = spark.createDataFrame(psi_results_df)

spark_psi_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("hackbricks.drift_metrics")
# mode="append" means each time you run this notebook, it ADDS rows
# rather than replacing them. This builds up a drift history over time.

print("\nDrift metrics saved to Delta table: hackbricks.drift_metrics")

# Verify it was saved
print("\nVerifying saved data:")
display(spark.sql("SELECT * FROM hackbricks.drift_metrics ORDER BY computed_at DESC LIMIT 10"))

PSI Results DataFrame:


feature_name,psi_value,status,threshold,drift_detected,n_expected,n_actual,computed_at,batch_name
age,0.0,STABLE - No action needed,0.2,false,307511,1001,2026-04-11T14:14:31.829568,german_credit_simulation
credit_amount,0.0,STABLE - No action needed,0.2,false,307511,1001,2026-04-11T14:14:31.829582,german_credit_simulation
duration_months,0.0,STABLE - No action needed,0.2,false,307511,1001,2026-04-11T14:14:31.829587,german_credit_simulation
installment_rate,8.2832,DRIFT DETECTED - Retrain model,0.2,true,307511,1001,2026-04-11T14:14:31.829592,german_credit_simulation
existing_credits,0.0,STABLE - No action needed,0.2,false,307511,1001,2026-04-11T14:14:31.829596,german_credit_simulation



Drift metrics saved to Delta table: hackbricks.drift_metrics

Verifying saved data:


feature_name,psi_value,status,threshold,drift_detected,n_expected,n_actual,computed_at,batch_name
existing_credits,0.0,STABLE - No action needed,0.2,false,307511,1001,2026-04-11T14:14:31.829596,german_credit_simulation
installment_rate,8.2832,DRIFT DETECTED - Retrain model,0.2,true,307511,1001,2026-04-11T14:14:31.829592,german_credit_simulation
duration_months,0.0,STABLE - No action needed,0.2,false,307511,1001,2026-04-11T14:14:31.829587,german_credit_simulation
credit_amount,0.0,STABLE - No action needed,0.2,false,307511,1001,2026-04-11T14:14:31.829582,german_credit_simulation
age,0.0,STABLE - No action needed,0.2,false,307511,1001,2026-04-11T14:14:31.829568,german_credit_simulation


In [0]:
mlflow.set_experiment("/Users/rajath2010rrp@gmail.com/hackbricks_credit_risk")

with mlflow.start_run(run_name="drift_detection_german_credit"):

    # Log each PSI value
    for r in psi_results:
        mlflow.log_metric(f"psi_{r['feature_name']}", r['psi_value'])

    mlflow.log_metric("n_drifted_features", len(drifted_features))
    mlflow.log_metric("max_psi", max(r['psi_value'] for r in psi_results))
    mlflow.log_metric("drift_threshold", 0.2)
    mlflow.log_param("batch_name", "german_credit_simulation")
    mlflow.log_param("n_expected_samples", len(train_df))
    mlflow.log_param("n_actual_samples", len(german_df))

    # Tag whether retraining should be triggered
    mlflow.set_tag("retrain_required", str(len(drifted_features) > 0))

print("\nPSI metrics logged to MLflow.")
print("\nNotebook 2 complete.")
print(f"Drift detected: {'YES - proceed to Notebook 3' if drifted_features else 'NO - model is stable'}")


PSI metrics logged to MLflow.

Notebook 2 complete.
Drift detected: YES - proceed to Notebook 3
